# FastMCP Server with Hugging Face Hub Resources

Build a FastMCP server that exposes Hugging Face Hub models and datasets as queryable MCP resources.

[Read the full article](https://jheiduk.com/posts/fastmcp-huggingface-hub/)

In [1]:
!pip install fastmcp huggingface_hub


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Querying Hugging Face Hub API

These cells demonstrate the raw `HfApi` calls that the MCP server wraps as resources. Run them standalone to inspect the data shape before adding the MCP layer.

In [2]:
from huggingface_hub import HfApi
import json

api = HfApi()

c:\Users\julie\jheiduk.com\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
models = list(api.list_models(filter="text-generation", sort="likes", limit=5))

for m in models:
    print(f"{m.id}: {m.likes or 0} likes, {m.downloads or 0} downloads")

deepseek-ai/DeepSeek-R1: 13021 likes, 733701 downloads
meta-llama/Meta-Llama-3-8B: 6466 likes, 1812133 downloads
meta-llama/Llama-3.1-8B-Instruct: 5491 likes, 6157607 downloads
bigscience/bloom: 4985 likes, 3539 downloads
meta-llama/Llama-2-7b-chat-hf: 4710 likes, 317632 downloads


In [4]:
# Fetch full metadata for a specific model
info = api.model_info("google/gemma-2-2b")

print(json.dumps({
    "id": info.id,
    "author": info.author,
    "task": info.pipeline_tag or "unknown",
    "likes": info.likes,
    "downloads": info.downloads,
    "tags": info.tags[:15],  # cap to avoid oversized output
}, indent=2))

{
  "id": "google/gemma-2-2b",
  "author": "google",
  "task": "text-generation",
  "likes": 627,
  "downloads": 164292,
  "tags": [
    "transformers",
    "safetensors",
    "gemma2",
    "text-generation",
    "arxiv:2009.03300",
    "arxiv:1905.07830",
    "arxiv:1911.11641",
    "arxiv:1904.09728",
    "arxiv:1905.10044",
    "arxiv:1907.10641",
    "arxiv:1811.00937",
    "arxiv:1809.02789",
    "arxiv:1911.01547",
    "arxiv:1705.03551",
    "arxiv:2107.03374"
  ]
}


In [5]:
models = list(api.list_datasets(sort="likes", limit=5))

for m in models:
    print(f"{m.id}: {m.likes or 0} likes, {m.downloads or 0} downloads")    

fka/prompts.chat: 9597 likes, 19023 downloads
HuggingFaceFW/fineweb: 2672 likes, 164066 downloads
Anthropic/hh-rlhf: 1663 likes, 17755 downloads
Open-Orca/OpenOrca: 1501 likes, 17921 downloads
OpenAssistant/oasst1: 1483 likes, 11473 downloads


In [6]:
# Fetch full metadata for a specific dataset
info = api.dataset_info("rajpurkar/squad")

print(json.dumps({
    "id": info.id,
    "author": info.author,
    "likes": info.likes,
    "downloads": info.downloads,
    "tags": info.tags[:15],
}, indent=2))

{
  "id": "rajpurkar/squad",
  "author": "rajpurkar",
  "likes": 354,
  "downloads": 76445,
  "tags": [
    "task_categories:question-answering",
    "task_ids:extractive-qa",
    "annotations_creators:crowdsourced",
    "language_creators:crowdsourced",
    "language_creators:found",
    "multilinguality:monolingual",
    "source_datasets:extended|wikipedia",
    "language:en",
    "license:cc-by-sa-4.0",
    "size_categories:10K<n<100K",
    "format:parquet",
    "modality:text",
    "library:datasets",
    "library:pandas",
    "library:mlcroissant"
  ]
}


## Building the FastMCP Server

This cell writes `hf_server.py` to disk — the complete FastMCP server with four resources: two static listing resources and two URI-template resources for individual model and dataset lookup.

In [7]:
server_code = '''
from fastmcp import FastMCP
from huggingface_hub import HfApi
import json

mcp = FastMCP("HuggingFace Hub Explorer")
api = HfApi()


@mcp.resource("hf://models")
def list_models() -> str:
    """Top 10 trending text-generation models on Hugging Face Hub."""
    models = list(api.list_models(filter="text-generation", sort="likes", direction=-1, limit=10))
    return json.dumps([
        {"id": m.id, "likes": m.likes or 0, "downloads": m.downloads or 0}
        for m in models
    ], indent=2)


@mcp.resource("hf://datasets")
def list_datasets() -> str:
    """Top 10 most liked datasets on Hugging Face Hub."""
    datasets = list(api.list_datasets(sort="likes", direction=-1, limit=10))
    return json.dumps([
        {"id": d.id, "likes": d.likes or 0, "downloads": d.downloads or 0}
        for d in datasets
    ], indent=2)


@mcp.resource("hf://models/{owner}/{name}")
def get_model(owner: str, name: str) -> str:
    """Fetch metadata for a specific model from Hugging Face Hub."""
    info = api.model_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "task": info.pipeline_tag or "unknown",
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


@mcp.resource("hf://datasets/{owner}/{name}")
def get_dataset(owner: str, name: str) -> str:
    """Fetch metadata for a specific dataset from Hugging Face Hub."""
    info = api.dataset_info(f"{owner}/{name}")
    return json.dumps({
        "id": info.id,
        "author": info.author,
        "likes": info.likes,
        "downloads": info.downloads,
        "tags": info.tags[:15],
    }, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("hf_server.py", "w") as f:
    f.write(server_code.strip())

print("Server written to hf_server.py")

Server written to hf_server.py


## Testing with the FastMCP Client

`Client("hf_server.py")` launches the server as a subprocess over stdio. The cells below write and execute a test client that reads all four resources. Make sure the server was written in the cell above before running.

In [8]:
client_lines = [
    "import asyncio",
    "from fastmcp import Client",
    "",
    "",
    "async def main():",
    '    async with Client("hf_server.py") as client:',
    "        # 1. Trending model catalogue",
    '        result = await client.read_resource("hf://models")',
    '        print("Trending models (first 400 chars):")',
    "        print(result[0].text[:400])",
    "",
    "        # 2. Specific model detail via URI template",
    '        result = await client.read_resource("hf://models/google/gemma-2-2b")',
    '        print("\\nModel detail — google/gemma-2-2b:")',
    "        print(result[0].text)",
    "",
    "        # 3. Trending dataset catalogue",
    '        result = await client.read_resource("hf://datasets")',
    '        print("\\nTrending datasets (first 400 chars):")',
    "        print(result[0].text[:400])",
    "",
    "        # 4. Specific dataset detail via URI template",
    '        result = await client.read_resource("hf://datasets/rajpurkar/squad")',
    '        print("\\nDataset detail — rajpurkar/squad:")',
    "        print(result[0].text)",
    "",
    "",
    "asyncio.run(main())",
]

with open("hf_client.py", "w", encoding="utf-8") as f:
    f.write("\n".join(client_lines) + "\n")

print("Client written to hf_client.py")

Client written to hf_client.py


In [9]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "hf_client.py"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Trending models (first 400 chars):
[
  {
    "id": "deepseek-ai/DeepSeek-R1",
    "likes": 13021,
    "downloads": 733701
  },
  {
    "id": "meta-llama/Meta-Llama-3-8B",
    "likes": 6466,
    "downloads": 1812133
  },
  {
    "id": "meta-llama/Llama-3.1-8B-Instruct",
    "likes": 5491,
    "downloads": 6157607
  },
  {
    "id": "bigscience/bloom",
    "likes": 4985,
    "downloads": 3539
  },
  {
    "id": "meta-llama/Llama-2-7b

Model detail â€” google/gemma-2-2b:
{
  "id": "google/gemma-2-2b",
  "author": "google",
  "task": "text-generation",
  "likes": 627,
  "downloads": 164292,
  "tags": [
    "transformers",
    "safetensors",
    "gemma2",
    "text-generation",
    "arxiv:2009.03300",
    "arxiv:1905.07830",
    "arxiv:1911.11641",
    "arxiv:1904.09728",
    "arxiv:1905.10044",
    "arxiv:1907.10641",
    "arxiv:1811.00937",
    "arxiv:1809.02789",
    "arxiv:1911.01547",
    "arxiv:1705.03551",
    "arxiv:2107.03374"
  ]
}

Trending datasets (first 400 chars):
[
  {
    "